# 05 · Final model: train once on all boreholes and freeze it

The model chosen in notebook 04 is tuned again on **all** boreholes of the area, with the same procedure as the inner CV (groups = the 5 blockCV folds), and then trained on all cells. The depth-only reference is trained the same way, for notebook 07.

The file is then **frozen**: its SHA-256 fingerprint is stored, and notebooks 06-08 refuse to run if the file changes. Running this notebook again is refused once a final model exists (set `REFIT = True` only with a reason you report in the thesis).

**Run time:** 5-20 minutes (GWRF/GWXGB are the slowest).

In [ ]:
import json, sys, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image

sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "settings.toml").exists())))
import helpers as H

S = H.load_settings()
T0 = time.time()
print("QUICK TEST RUN - not thesis results" if S["run"]["quick"] else "FULL RUN", "| area:", S["area"]["area"])
import joblib

REFIT = False
cells = pd.read_csv(H.processed_dir(S) / "cells.csv")
folds, block_m = H.load_folds(S, 1)
sel = json.loads((H.out_dir(S, "04_results") / "selection.json").read_text())
NAME = sel["selected"]
OUT, PRIV = H.out_dir(S, "05_final_model"), H.out_dir(S, "05_final_model", private=True)
record = OUT / "final_model.json"
if record.exists() and not REFIT:
    raise RuntimeError(f"A final model is already frozen ({json.loads(record.read_text())['model']}). "
                       "Set REFIT = True only with a documented reason.")
print("Final model:", NAME, f"| block size {block_m:.1f} m | {cells.BH_Code.nunique()} boreholes, {len(cells)} cells")

In [ ]:
groups = cells.BH_Code.map(folds).values
final = {}
for name in (NAME, H.DEPTH_ONLY):
    seed = H.seed_of(S, "final", name)
    best = H.tune(name, cells, groups, seed, S)
    model = H.make_model(name, best["params"], seed, S).fit(cells, H.fill_weights(cells, best["params"]))
    if hasattr(model, "local_"):          # GW models: train every local model now, so the frozen file is complete
        model.p_clay(cells)
    path = PRIV / f"final_{name}.joblib"
    joblib.dump(model, path, compress=3)
    final[name] = {"model": name, "params": best["params"], "threshold": best["threshold"],
                   "inner_macro_f1": best["inner_macro_f1"], "file": path.name, "sha256": H.sha256(path)}
    best["trials"].to_csv(OUT / f"final_trials_{name}.csv", index=False)
    print(f"{name}: threshold {best['threshold']:.2f}, inner Macro-F1 {best['inner_macro_f1']:.3f}, "
          f"SHA-256 {final[name]['sha256'][:16]}")

In [ ]:
rec = {**final[NAME], "depth_only": final[H.DEPTH_ONLY], "block_m": block_m, "area": S["area"]["area"],
       "train_boreholes": int(cells.BH_Code.nunique()), "train_cells": len(cells),
       "train_filled_cells": int((cells.measured == 0).sum()),
       "top_elevation_range_m": [float(cells.Top_Elevation_m.min()), float(cells.Top_Elevation_m.max())],
       "frozen_on": time.strftime("%Y-%m-%d %H:%M"), "quick": S["run"]["quick"]}
record.write_text(json.dumps(rec, indent=1, default=str))
print(json.dumps(rec, indent=1, default=str))
print(f"Finished in {(time.time() - T0) / 60:.1f} min. Next: notebook 06.")